# 🏋️ Dumbbell Romanian Deadlift (RDL) Form Analyzer
### CAI2840C — Introduction to Computer Vision | Michael B. Bowman

---

This notebook uses **Google MediaPipe Pose** to analyze still images of the two-handed dumbbell Romanian Deadlift.  
Rather than labeling form as simply "good" or "bad", the system:
- Detects key body landmarks (joints)
- Calculates joint angles critical to RDL mechanics
- Flags specific form issues (e.g. rounded back, squat-pattern knees)
- Provides a **corrective cue** for each issue found

**Key angles analyzed:**
| Angle | Landmark Points | What it tells us |
|---|---|---|
| Hip Hinge Angle | Shoulder → Hip → Knee | Depth and quality of hip hinge |
| Spine / Torso Angle | Shoulder → Hip (vs. horizontal) | Back flatness / rounding |
| Knee Flexion Angle | Hip → Knee → Ankle | Whether it looks like an RDL vs. a squat |

---
## Cell 1 — Install & Import Dependencies

In [ ]:
# ─────────────────────────────────────────────
# CELL 1 — Install & Import Dependencies
# ─────────────────────────────────────────────
# Uses the new MediaPipe Tasks API (compatible with 0.10.13+)
# No version pinning needed — works with whatever Colab has installed.

!pip install mediapipe --quiet
!pip install protobuf --upgrade --quiet

import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display
import math
import urllib.request
import os
from PIL import Image

# Download the pose landmarker model file (required by Tasks API)
MODEL_PATH = "pose_landmarker_heavy.task"
if not os.path.exists(MODEL_PATH):
    print("Downloading pose landmarker model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/latest/pose_landmarker_heavy.task",
        MODEL_PATH
    )
    print("✅ Model downloaded.")
else:
    print("✅ Model already present.")

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions, RunningMode
from mediapipe import Image as MpImage, ImageFormat

print(f"✅ All libraries imported successfully.")
print(f"   MediaPipe version: {mp.__version__}")
print(f"   OpenCV version:    {cv2.__version__}")


---
## Cell 2 — Core Helper Functions
These functions do the geometry. `calculate_angle()` is the workhorse — it takes three (x,y) points and returns the angle at the middle point in degrees.

In [ ]:
# ─────────────────────────────────────────────
# CELL 2 — Core Helper Functions
# ─────────────────────────────────────────────

def calculate_angle(a, b, c):
    """
    Calculate the angle (in degrees) at point B, given three points A, B, C.
    Uses the dot product of vectors BA and BC.

    Args:
        a (tuple): (x, y) of the first point
        b (tuple): (x, y) of the vertex point (angle is measured HERE)
        c (tuple): (x, y) of the third point

    Returns:
        float: Angle in degrees (0–180)
    """
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    # Vectors from B to A, and from B to C
    ba = a - b
    bc = c - b

    # Cosine of the angle using dot product formula
    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
    cosine_angle = np.clip(cosine_angle, -1.0, 1.0)  # guard against float errors

    angle = math.degrees(math.acos(cosine_angle))
    return round(angle, 1)


def calculate_torso_angle(shoulder, hip):
    """
    Calculate the angle of the torso relative to vertical (0° = perfectly upright).
    Uses the vector from hip to shoulder vs. the vertical axis.

    Args:
        shoulder (tuple): (x, y) shoulder landmark
        hip (tuple): (x, y) hip landmark

    Returns:
        float: Angle in degrees from vertical (0° = standing upright)
    """
    dx = shoulder[0] - hip[0]
    dy = shoulder[1] - hip[1]  # Note: y increases downward in image coords

    # Angle from vertical (straight up = 0°)
    angle_from_vertical = math.degrees(math.atan2(abs(dx), abs(dy)))
    return round(angle_from_vertical, 1)


def get_landmark_coords(landmarks, landmark_name, image_width, image_height):
    """
    Extract (x, y) pixel coordinates for a named MediaPipe landmark.

    Args:
        landmarks: MediaPipe pose landmark object
        landmark_name (str): Name of the landmark (e.g. 'LEFT_SHOULDER')
        image_width (int): Width of the image in pixels
        image_height (int): Height of the image in pixels

    Returns:
        tuple: (x_pixel, y_pixel)
    """
    lm = landmarks[mp.solutions.pose.PoseLandmark[landmark_name].value]
    return (int(lm.x * image_width), int(lm.y * image_height))


print("✅ Helper functions defined.")

---
## Cell 3 — Form Assessment Rules
This is the "expert system" layer. Each rule checks one angle against a threshold and returns a finding dict with the angle name, the measured value, a diagnosis, and a corrective cue.

In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Form Assessment Rules
# ─────────────────────────────────────────────
# Each rule function returns a dict with:
#   'angle_name'  : label for display
#   'value'       : measured angle in degrees
#   'status'      : 'OK', 'WARNING', or 'FLAG'
#   'issue'       : description of what was found (None if OK)
#   'correction'  : coaching cue to fix it (None if OK)

# ── THRESHOLDS ──────────────────────────────────────────────────────────
# These are based on established biomechanical guidelines for the RDL.
# You can adjust them as you refine the model.

# Hip hinge angle (shoulder–hip–knee)
#   In a proper RDL hinge: roughly 60°–130° at the bottom
#   Too large (>130°) = barely hinging, standing too upright
#   Too small (<60°)  = squatting pattern, not hinging
HIP_ANGLE_MIN = 60
HIP_ANGLE_MAX = 130

# Spine/torso lean (degrees from vertical)
#   Proper RDL hinge = torso 25°–75° from vertical at bottom
#   <25° = not enough hinge (too upright)
#   >75° = excessively horizontal / possible back rounding compensation
TORSO_LEAN_MIN = 25
TORSO_LEAN_MAX = 75

# Knee flexion angle (hip–knee–ankle)
#   RDL = soft bend, knees ~130°–170° (nearly straight)
#   <130° = too much knee bend → becoming a squat, not an RDL
KNEE_ANGLE_MIN = 130

# ────────────────────────────────────────────────────────────────────────

def assess_hip_hinge(hip_angle, is_hinge_phase):
    """
    Evaluate the hip hinge angle (shoulder–hip–knee).
    Only meaningful when the person is in the hinge/bottom phase.
    """
    result = {
        'angle_name': 'Hip Hinge Angle (Shoulder→Hip→Knee)',
        'value': hip_angle,
        'status': 'OK',
        'issue': None,
        'correction': None
    }

    if not is_hinge_phase:
        result['status'] = 'INFO'
        result['issue'] = 'Person appears to be in standing/start position — hip hinge assessment skipped.'
        return result

    if hip_angle < HIP_ANGLE_MIN:
        result['status'] = 'FLAG'
        result['issue'] = f'Hip angle is {hip_angle}° — too small. This looks more like a squat than an RDL.'
        result['correction'] = (
            'Push your hips BACK (not down). Think "closing a car door with your hips." '
            'Keep your shins more vertical and send the dumbbells down your legs.'
        )
    elif hip_angle > HIP_ANGLE_MAX:
        result['status'] = 'WARNING'
        result['issue'] = f'Hip angle is {hip_angle}° — not enough hip hinge depth.'
        result['correction'] = (
            'Hinge deeper at the hips. Keep a slight knee bend, push your hips back further, '
            'and lower the dumbbells toward mid-shin or the floor.'
        )
    else:
        result['issue'] = f'Hip angle {hip_angle}° is within the expected RDL range ({HIP_ANGLE_MIN}°–{HIP_ANGLE_MAX}°).'

    return result


def assess_spine_angle(torso_angle, is_hinge_phase):
    """
    Evaluate the torso lean (degrees from vertical).
    Tells us if the back is flat and appropriately inclined.
    """
    result = {
        'angle_name': 'Torso / Spine Lean (° from vertical)',
        'value': torso_angle,
        'status': 'OK',
        'issue': None,
        'correction': None
    }

    if not is_hinge_phase:
        result['status'] = 'INFO'
        result['issue'] = 'Standing position — torso should be upright. Hinge cues not applicable.'
        return result

    if torso_angle < TORSO_LEAN_MIN:
        result['status'] = 'WARNING'
        result['issue'] = f'Torso angle is only {torso_angle}° from vertical — insufficient forward lean.'
        result['correction'] = (
            'Hinge more at the hips. Push hips back and allow your torso to incline forward '
            'while keeping your back flat and chest up.'
        )
    elif torso_angle > TORSO_LEAN_MAX:
        result['status'] = 'FLAG'
        result['issue'] = f'Torso angle is {torso_angle}° from vertical — excessively horizontal. May indicate back rounding.'
        result['correction'] = (
            'Focus on keeping your chest lifted and shoulder blades pulled together. '
            'Do not let your upper back round or your shoulders collapse forward. '
            'Imagine a straight line from your head to your tailbone.'
        )
    else:
        result['issue'] = f'Torso lean {torso_angle}° is within the expected range ({TORSO_LEAN_MIN}°–{TORSO_LEAN_MAX}°).'

    return result


def assess_knee_angle(knee_angle, is_hinge_phase):
    """
    Evaluate the knee flexion angle (hip–knee–ankle).
    In an RDL, knees should stay mostly extended (soft bend only).
    """
    result = {
        'angle_name': 'Knee Flexion Angle (Hip→Knee→Ankle)',
        'value': knee_angle,
        'status': 'OK',
        'issue': None,
        'correction': None
    }

    if not is_hinge_phase:
        result['status'] = 'INFO'
        result['issue'] = 'Standing position — knee angle assessment most useful at the bottom of the hinge.'
        return result

    if knee_angle < KNEE_ANGLE_MIN:
        result['status'] = 'FLAG'
        result['issue'] = f'Knee angle is {knee_angle}° — knees are bending too much. This is a squat pattern, not an RDL.'
        result['correction'] = (
            'Keep a soft, FIXED bend in the knees — they should not bend further as you lower. '
            'Drive movement from the hips, not the knees. Lock a slight bend in place and maintain it throughout.'
        )
    else:
        result['issue'] = f'Knee angle {knee_angle}° indicates appropriate soft-bend (>{KNEE_ANGLE_MIN}°). Good RDL knee position.'

    return result


print("✅ Assessment rules defined.")
print(f"   Hip hinge range:   {HIP_ANGLE_MIN}° – {HIP_ANGLE_MAX}°")
print(f"   Torso lean range:  {TORSO_LEAN_MIN}° – {TORSO_LEAN_MAX}° from vertical")
print(f"   Knee min angle:    >{KNEE_ANGLE_MIN}° (soft bend, not a squat)")

---
## Cell 4 — Visualization & Report Functions
These functions draw the skeleton overlay on the image and print the structured feedback report.

In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Visualization & Report Functions
# ─────────────────────────────────────────────

# Color scheme (BGR format for OpenCV)
COLOR_OK      = (50, 205, 50)    # Green  — within normal range
COLOR_WARNING = (0, 165, 255)    # Orange — borderline
COLOR_FLAG    = (0, 0, 220)      # Red    — needs correction
COLOR_INFO    = (200, 200, 200)  # Gray   — informational
COLOR_JOINT   = (255, 255, 255)  # White  — joint dots
COLOR_TEXT_BG = (30, 30, 30)     # Dark   — text background


def status_color(status):
    """Map a status string to its BGR color."""
    return {
        'OK': COLOR_OK,
        'WARNING': COLOR_WARNING,
        'FLAG': COLOR_FLAG,
        'INFO': COLOR_INFO
    }.get(status, COLOR_INFO)


def draw_angle_arc(image, vertex, point_a, point_c, angle_value, color, label):
    """
    Draw a line from vertex to both points and display the angle value as text.
    """
    cv2.line(image, vertex, point_a, color, 2)
    cv2.line(image, vertex, point_c, color, 2)
    cv2.circle(image, vertex, 8, color, -1)

    # Text label near the vertex
    text = f"{label}: {angle_value}°"
    tx, ty = vertex[0] + 12, vertex[1] - 10
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
    cv2.rectangle(image, (tx - 3, ty - th - 3), (tx + tw + 3, ty + 5), COLOR_TEXT_BG, -1)
    cv2.putText(image, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1, cv2.LINE_AA)


def draw_skeleton_overlay(image, landmarks, angles_data, image_width, image_height):
    """
    Draw the MediaPipe pose skeleton and overlay key angle measurements.

    Args:
        image: OpenCV BGR image (will be modified in-place)
        landmarks: MediaPipe pose landmarks
        angles_data: dict of computed angles and their assessment results
        image_width, image_height: image dimensions
    """
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils
    mp_drawing_styles = mp.solutions.drawing_styles

    # Draw the full MediaPipe skeleton
    mp_drawing.draw_landmarks(
        image,
        landmarks,
        mp_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style()
    )

    lm = landmarks.landmark
    side = angles_data.get('side', 'LEFT')  # Which side of body was used

    def coords(name):
        return get_landmark_coords(lm, name, image_width, image_height)

    # Pull landmark positions for the analyzed side
    shoulder = coords(f'{side}_SHOULDER')
    hip      = coords(f'{side}_HIP')
    knee     = coords(f'{side}_KNEE')
    ankle    = coords(f'{side}_ANKLE')

    # Draw angle overlays
    hip_result   = angles_data.get('hip_result', {})
    knee_result  = angles_data.get('knee_result', {})
    torso_result = angles_data.get('torso_result', {})

    draw_angle_arc(image, hip,   shoulder, knee,   angles_data.get('hip_angle', 0),
                   status_color(hip_result.get('status', 'INFO')), 'Hip')
    draw_angle_arc(image, knee,  hip,      ankle,  angles_data.get('knee_angle', 0),
                   status_color(knee_result.get('status', 'INFO')), 'Knee')
    draw_angle_arc(image, shoulder, hip,   shoulder, angles_data.get('torso_angle', 0),
                   status_color(torso_result.get('status', 'INFO')), 'Torso')

    # Phase label (top-left corner)
    phase_label = "PHASE: HINGE" if angles_data.get('is_hinge_phase') else "PHASE: STANDING"
    cv2.rectangle(image, (8, 8), (260, 38), COLOR_TEXT_BG, -1)
    cv2.putText(image, phase_label, (12, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 100), 2, cv2.LINE_AA)

    return image


def print_report(angles_data, filename=""):
    """
    Print a structured text report of all angle assessments to the notebook output.
    """
    separator = "═" * 65
    thin_sep  = "─" * 65

    status_icons = {'OK': '✅', 'WARNING': '⚠️ ', 'FLAG': '🚩', 'INFO': 'ℹ️ '}

    print(f"\n{separator}")
    print(f"  🏋️  RDL FORM ANALYSIS REPORT")
    if filename:
        print(f"  📷 Image: {filename}")
    print(f"  🔍 Side analyzed: {angles_data.get('side', 'LEFT')}")
    print(f"  📐 Phase detected: {'HINGE / BOTTOM' if angles_data.get('is_hinge_phase') else 'STANDING / START'}")
    print(separator)

    for key in ['torso_result', 'hip_result', 'knee_result']:
        r = angles_data.get(key, {})
        if not r:
            continue
        icon = status_icons.get(r.get('status', 'INFO'), 'ℹ️ ')
        print(f"\n{icon}  {r.get('angle_name', '')}")
        print(f"   Measured value : {r.get('value', 'N/A')}°")
        print(f"   Finding        : {r.get('issue', 'No issue noted.')}")
        if r.get('correction'):
            print(f"   ➡️  Correction   : {r['correction']}")
        print(thin_sep)

    # Summary
    flags    = [r for k in ['torso_result','hip_result','knee_result']
                if (r := angles_data.get(k, {})) and r.get('status') == 'FLAG']
    warnings = [r for k in ['torso_result','hip_result','knee_result']
                if (r := angles_data.get(k, {})) and r.get('status') == 'WARNING']

    print(f"\n  SUMMARY: {len(flags)} issue(s) flagged | {len(warnings)} warning(s)")
    if not flags and not warnings:
        print("  🎉 No major form issues detected for this phase!")
    print(separator)


print("✅ Visualization and report functions defined.")

---
## Cell 5 — Main Analysis Pipeline
This is the function that ties everything together. It takes an image, runs MediaPipe pose detection, computes all angles, runs the assessment rules, draws the overlay, and prints the report.

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Main Analysis Pipeline (Tasks API)
# ─────────────────────────────────────────────

def get_coords(landmark, w, h):
    """Convert normalized landmark to pixel coordinates."""
    return (int(landmark.x * w), int(landmark.y * h))

def detect_side_tasks(landmarks, w, h):
    """Pick the more visible side using visibility scores."""
    # Indices: left shoulder=11, left hip=23, left knee=25, left ankle=27
    #          right shoulder=12, right hip=24, right knee=26, right ankle=28
    left_vis  = sum(landmarks[i].visibility for i in [11, 23, 25, 27])
    right_vis = sum(landmarks[i].visibility for i in [12, 24, 26, 28])
    return 'LEFT' if left_vis >= right_vis else 'RIGHT'

# Side index maps
SIDE_IDX = {
    'LEFT':  {'shoulder': 11, 'hip': 23, 'knee': 25, 'ankle': 27},
    'RIGHT': {'shoulder': 12, 'hip': 24, 'knee': 26, 'ankle': 28},
}

def analyze_rdl_image(image_bgr, filename="image"):
    """
    Full analysis pipeline using MediaPipe Tasks PoseLandmarker.
    """
    if image_bgr is None:
        print(f"❌ Could not load image: {filename}")
        return None, None

    # Resize if too large
    max_dim = 900
    h, w = image_bgr.shape[:2]
    if max(h, w) > max_dim:
        scale = max_dim / max(h, w)
        image_bgr = cv2.resize(image_bgr, (int(w * scale), int(h * scale)))
    h, w = image_bgr.shape[:2]

    # Convert to RGB for MediaPipe
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image  = MpImage(image_format=ImageFormat.SRGB, data=image_rgb)

    # ── Run PoseLandmarker ──────────────────────────────────────────
    options = PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=RunningMode.IMAGE,
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_segmentation_masks=False
    )

    with PoseLandmarker.create_from_options(options) as detector:
        result = detector.detect(mp_image)

    if not result.pose_landmarks or len(result.pose_landmarks) == 0:
        print(f"❌ No person detected in '{filename}'. Tips:")
        print("   • Ensure full body (head to feet) is visible")
        print("   • Use a clear side-view image")
        print("   • Try better lighting or less cluttered background")
        return image_bgr, None

    landmarks = result.pose_landmarks[0]  # First (only) person

    # ── Detect side & extract coords ────────────────────────────────
    side = detect_side_tasks(landmarks, w, h)
    idx  = SIDE_IDX[side]

    shoulder = get_coords(landmarks[idx['shoulder']], w, h)
    hip      = get_coords(landmarks[idx['hip']],      w, h)
    knee     = get_coords(landmarks[idx['knee']],     w, h)
    ankle    = get_coords(landmarks[idx['ankle']],    w, h)

    # ── Calculate angles ────────────────────────────────────────────
    hip_angle   = calculate_angle(shoulder, hip, knee)
    knee_angle  = calculate_angle(hip, knee, ankle)
    torso_angle = calculate_torso_angle(shoulder, hip)

    # ── Detect phase ────────────────────────────────────────────────
    is_hinge_phase = torso_angle > 20

    # ── Assess form ─────────────────────────────────────────────────
    hip_result   = assess_hip_hinge(hip_angle,  is_hinge_phase)
    torso_result = assess_spine_angle(torso_angle, is_hinge_phase)
    knee_result  = assess_knee_angle(knee_angle, is_hinge_phase)

    angles_data = {
        'side':           side,
        'is_hinge_phase': is_hinge_phase,
        'hip_angle':      hip_angle,
        'knee_angle':     knee_angle,
        'torso_angle':    torso_angle,
        'hip_result':     hip_result,
        'torso_result':   torso_result,
        'knee_result':    knee_result,
    }

    # ── Draw skeleton overlay manually ──────────────────────────────
    annotated = image_bgr.copy()

    # Draw key joint connections
    connections = [
        (landmarks[idx['shoulder']], landmarks[idx['hip']]),
        (landmarks[idx['hip']],      landmarks[idx['knee']]),
        (landmarks[idx['knee']],     landmarks[idx['ankle']]),
    ]
    for a, b in connections:
        pt1 = get_coords(a, w, h)
        pt2 = get_coords(b, w, h)
        cv2.line(annotated, pt1, pt2, (0, 255, 100), 3)

    # Draw joint dots
    for pt in [shoulder, hip, knee, ankle]:
        cv2.circle(annotated, pt, 8, (255, 255, 255), -1)
        cv2.circle(annotated, pt, 8, (0, 200, 80), 2)

    # Draw angle labels
    def draw_label(img, point, label, value, status):
        color = {
            'OK': (50, 205, 50), 'WARNING': (0, 165, 255),
            'FLAG': (0, 0, 220), 'INFO': (180, 180, 180)
        }.get(status, (180, 180, 180))
        text = f"{label}: {value}deg"
        tx, ty = point[0] + 12, point[1] - 10
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(img, (tx-3, ty-th-3), (tx+tw+3, ty+5), (30,30,30), -1)
        cv2.putText(img, text, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 1, cv2.LINE_AA)

    draw_label(annotated, hip,      'Hip',   hip_angle,   hip_result.get('status','INFO'))
    draw_label(annotated, knee,     'Knee',  knee_angle,  knee_result.get('status','INFO'))
    draw_label(annotated, shoulder, 'Torso', torso_angle, torso_result.get('status','INFO'))

    # Phase label
    phase_label = "PHASE: HINGE" if is_hinge_phase else "PHASE: STANDING"
    cv2.rectangle(annotated, (8, 8), (260, 38), (30,30,30), -1)
    cv2.putText(annotated, phase_label, (12, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 100), 2, cv2.LINE_AA)

    # ── Print report ────────────────────────────────────────────────
    print_report(angles_data, filename=filename)

    return annotated, angles_data

print("✅ Main analysis pipeline defined (Tasks API).")


In [ ]:
# ─────────────────────────────────────────────
# FIX — Properly install & verify poppler
# ─────────────────────────────────────────────
!apt-get update -qq
!apt-get install -y poppler-utils

# Verify it's actually installed and on PATH
import shutil
path = shutil.which("pdftoppm")
if path:
    print(f"✅ poppler is installed at: {path}")
else:
    print("❌ poppler still not found — see error output above")

---
## Cell 6 — Run Analysis on Uploaded Image(s)
Upload one or more images **or a PDF** containing multiple images. Each image/page will be analyzed and displayed with its annotated skeleton overlay.

**Supported formats:** `.jpg`, `.jpeg`, `.png`, `.pdf`

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Upload & Analyze Image(s) or PDF
# ─────────────────────────────────────────────
# Accepts: .jpg / .jpeg / .png  →  analyzed directly
#          .pdf                 →  each page is converted to an image first
#
# A file picker will appear. You can select multiple files at once.

!pip install pdf2image --quiet
!apt-get install -y poppler-utils --quiet > /dev/null 2>&1

from pdf2image import convert_from_bytes

print("📂 Please select one or more files to analyze (images or PDF)...")
uploaded = files.upload()  # Opens Colab's file picker

results_store = {}  # Store results for all images across all uploads


def process_and_display(image_bgr, label):
    """Run analysis on one BGR image and display results."""
    annotated, angles_data = analyze_rdl_image(image_bgr, filename=label)

    if annotated is not None:
        results_store[label] = {
            'annotated': annotated,
            'original':  image_bgr,
            'angles':    angles_data
        }

        orig_rgb  = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        annot_rgb = cv2.cvtColor(annotated,  cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(orig_rgb);  axes[0].set_title(f"Original — {label}", fontsize=11); axes[0].axis('off')
        axes[1].imshow(annot_rgb); axes[1].set_title("Annotated (Skeleton + Angles)", fontsize=11); axes[1].axis('off')
        plt.tight_layout()
        plt.show()


for filename, file_bytes in uploaded.items():
    print(f"\n{'═'*65}")
    print(f"  Uploaded: {filename}")
    print(f"{'═'*65}")

    ext = filename.lower().rsplit('.', 1)[-1]

    # ── PDF → convert each page to an image ───────────────────────────
    if ext == 'pdf':
        print(f"  📄 PDF detected — converting pages to images...")
        try:
            pil_pages = convert_from_bytes(file_bytes, dpi=200)
            print(f"  ✅ Found {len(pil_pages)} page(s) in PDF.")
        except Exception as e:
            print(f"  ❌ Could not convert PDF: {e}")
            continue

        for page_num, pil_img in enumerate(pil_pages, start=1):
            label = f"{filename} — page {page_num}"
            print(f"\n  Processing page {page_num}/{len(pil_pages)}: {label}")

            # Convert PIL (RGB) → numpy → BGR for OpenCV
            img_rgb = np.array(pil_img)
            image_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
            process_and_display(image_bgr, label)

    # ── Standard image (jpg / png / etc.) ─────────────────────────────
    else:
        nparr    = np.frombuffer(file_bytes, np.uint8)
        image_bgr = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        if image_bgr is None:
            print(f"  ❌ Could not decode '{filename}'. Make sure it is a valid image or PDF.")
            continue

        process_and_display(image_bgr, filename)

print(f"\n✅ Done. {len(results_store)} image(s) analyzed successfully.")


---
## Cell 7 — Batch Summary Table
After analyzing multiple images, this cell prints a clean comparison table showing all measured angles across images — useful for comparing proper vs. improper form examples side-by-side.

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Batch Summary Comparison Table
# ─────────────────────────────────────────────
# Run this after Cell 6 to see all images compared in one table.

import pandas as pd  # pandas makes the table easy to read in Colab

if not results_store:
    print("⚠️  No results yet — run Cell 6 first.")
else:
    rows = []
    for fname, data in results_store.items():
        a = data.get('angles', {})
        if not a:
            continue

        def flag(result_key):
            """Return a status emoji for a given result."""
            s = a.get(result_key, {}).get('status', 'INFO')
            return {'OK': '✅ OK', 'WARNING': '⚠️  Warn', 'FLAG': '🚩 Flag', 'INFO': 'ℹ️  Info'}.get(s, '')

        rows.append({
            'Image': fname,
            'Phase': 'Hinge' if a.get('is_hinge_phase') else 'Standing',
            'Side': a.get('side', '?'),
            'Hip Angle (°)': a.get('hip_angle', '—'),
            'Hip Status': flag('hip_result'),
            'Knee Angle (°)': a.get('knee_angle', '—'),
            'Knee Status': flag('knee_result'),
            'Torso Lean (°)': a.get('torso_angle', '—'),
            'Torso Status': flag('torso_result'),
        })

    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 40)
    print("\n📊 BATCH SUMMARY — All Analyzed Images")
    print("="*80)
    display(df)
    print("\nThreshold reference:")
    print(f"  Hip Hinge: {HIP_ANGLE_MIN}°–{HIP_ANGLE_MAX}° | Torso Lean: {TORSO_LEAN_MIN}°–{TORSO_LEAN_MAX}° | Knee: >{KNEE_ANGLE_MIN}°")

---
## Cell 8 — (Optional) Save Annotated Images
Download all annotated images to your local machine.

In [ ]:
for fname, data in results_store.items():
    annotated = data.get('annotated')
    if annotated is None:
        continue

    # Build a safe filename — replace spaces, dashes, dots with underscores
    safe_name = fname.replace(' ', '_').replace('—', '').replace('.', '_').replace('__','_').strip('_')
    out_name = f"{safe_name}_annotated.jpg"

    cv2.imwrite(out_name, annotated)
    print(f"💾 Saving: {out_name}")
    files.download(out_name)

print(f"\n✅ All {len(results_store)} annotated image(s) downloaded.")

---
## Cell 9 — (Optional) Adjust Thresholds & Re-Run
If you want to experiment with different angle thresholds (e.g. for a more or less strict classifier), change the values here and re-run Cells 3 → 6.

In [ ]:
# ─────────────────────────────────────────────
# CELL 9 — Adjust Thresholds (Optional)
# ─────────────────────────────────────────────
# Modify these values then re-run Cell 3 to apply new thresholds.
# Then re-run Cell 6 to re-analyze with the updated rules.

# ── Hip Hinge (shoulder–hip–knee angle) ────────────────────────────────
# Proper RDL hinge at bottom: ~70°–120°
# Decrease HIP_ANGLE_MIN if you want to allow deeper hinges
# Increase HIP_ANGLE_MAX if you want to allow less hinge depth
HIP_ANGLE_MIN = 60    # degrees — below this = too squat-like
HIP_ANGLE_MAX = 130   # degrees — above this = not hinging enough

# ── Torso Lean (degrees from vertical) ────────────────────────────────
# More lean = more hip dominant movement
# Decrease TORSO_LEAN_MAX for stricter back flatness enforcement
TORSO_LEAN_MIN = 25   # degrees — below this = too upright
TORSO_LEAN_MAX = 75   # degrees — above this = too horizontal / risk of rounding

# ── Knee Flexion ────────────────────────────────────────────────────────
# RDL = mostly straight legs (soft bend)
# Increase KNEE_ANGLE_MIN to require straighter legs
KNEE_ANGLE_MIN = 130  # degrees — below this = too much knee bend (squat pattern)

print("✅ Thresholds updated.")
print("   Now re-run Cell 3 to apply, then re-run Cell 6 to re-analyze.")
print(f"   Hip: {HIP_ANGLE_MIN}°–{HIP_ANGLE_MAX}° | Torso: {TORSO_LEAN_MIN}°–{TORSO_LEAN_MAX}° | Knee: >{KNEE_ANGLE_MIN}°")

## Appendix — MediaPipe Landmark Reference

MediaPipe detects **33 body landmarks**. The ones used in this notebook:

| Landmark | Index | Role in this analysis |
|---|---|---|
| LEFT/RIGHT_SHOULDER | 11/12 | Top of the torso line; hip hinge & torso angle |
| LEFT/RIGHT_HIP | 23/24 | Vertex of the hip hinge angle |
| LEFT/RIGHT_KNEE | 25/26 | Vertex of the knee flexion angle |
| LEFT/RIGHT_ANKLE | 27/28 | Bottom of the knee angle |

Full landmark map: https://developers.google.com/mediapipe/solutions/vision/pose_landmarker

---

## References

Bazarevsky, V., Grishchenko, I., Raveendran, K., Zhu, T., Zhang, F., & Grunderman, M. (2020, June 17). *BlazePose: On-device Real-time Body Pose tracking*. https://arxiv.org/pdf/2006.10204

Google. (2023). *MediaPipe Pose Landmarker*. https://developers.google.com/mediapipe/solutions/vision/pose_landmarker

Lee, S., Schultz, J., Timgren, J., Staelgraeve, K., Miller, M.,, & Liu, Y. (2018). An electromyographic and kinetic comparison of conventional and Romanian deadlifts. *Journal of Exercise Science & Fitness, 16*(3), 87–93. https://doi.org/10.1016/j.jesf.2018.08.001

Piper, T. J., & Waller, M. A. (2001). Variations of the deadlift. *Strength and Conditioning Journal, 23*(3), 66–73. https://paulogentil.com/pdf/TREINO%20DE%20FORC%CC%A7A/Treinamento%20com%20pesos/P10.pdf

Tomko, J. (n.d.). *How to do a dumbbell Romanian deadlift*. Men's Health. https://www.menshealth.com/fitness/a43720733/how-to-do-dumbbell-romanian-deadlift/

Turner, M., Appiah, K., & Kwok, S. C. (2024, April 19). *A Mobile-Phone Pose Estimation for Gym-Exercise Form Correction*. https://eprints.whiterose.ac.uk/id/eprint/210366/1/VISAPP_2024_266_CR.pdf